# SOXX 2019–2022 NPORT-P 申报索引原件

由已核对的 S000004354 series 检索页冻结 13 条原始 filing index URL；只存 SEC 索引字节和时钟，不读取 XML/持仓或收益。


In [1]:
import json,hashlib,subprocess,time,tarfile,tempfile,shlex
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_soxx_2019_2022_nport_filing_metadata_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):
 h=hashlib.sha256()
 with Path(p).open('rb') as f:
  for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
 return h.hexdigest()
assert plan['frozen'] and not (card/'attempt_started.json').exists()
assert sha(root/plan['parent_source']['series_index_parse'])==plan['parent_source']['series_index_parse_sha256']
raw=root/'data/snapshots/stage1_soxx_2019_2022_nport_filing_metadata_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','GET_attempts':0,'primary_document_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];error=None
for i,spec in enumerate(plan['exact_index_pages']):
 state['GET_attempts']+=1;(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
 accession=spec['accession'];body=raw/(accession+'.html');headers=raw/(accession+'.headers');started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(plan['transport']['connect_timeout_seconds']),'--max-time',str(plan['transport']['total_timeout_seconds']),'--max-filesize',str(plan['transport']['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',plan['transport']['user_agent'],'-w','%{http_code} %{url_effective}',spec['index_url']]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=60);fields=run.stdout.rsplit(' ',1);status=int(fields[0]) if len(fields)==2 and fields[0].isdigit() else None
  receipt={'accession':accession,'expected_filed':spec['filed_from_series_search'],'url':spec['index_url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':fields[-1] if fields else None,'stderr':run.stderr[:300]}
 except Exception as exc:
  receipt={'accession':accession,'expected_filed':spec['filed_from_series_search'],'url':spec['index_url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
 receipts.append(receipt)
 if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists() or body.stat().st_size>plan['transport']['max_response_bytes_each']:
  error='transport_or_size_failure';break
 if i+1<len(plan['exact_index_pages']):time.sleep(plan['transport']['seconds_between_requests'])
(card/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
manifest={'plan_sha256':sha(card/'plan.json'),'files':[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw.iterdir()) if p.is_file()],'capture_receipts_sha256':sha(card/'capture_receipts.json')}
(card/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='soxx-nport-index-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for x in manifest['files']:out.add(raw/x['name'],arcname='raw/'+x['name'])
  out.add(card/'source_manifest.json',arcname='source_manifest.json');out.add(card/'capture_receipts.json',arcname='capture_receipts.json')
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='soxx-nport-index-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:
   run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as source:
   for x in manifest['files']:
    member=source.extractfile('raw/'+x['name']);assert member and hashlib.sha256(member.read()).hexdigest()==x['sha256']
result={'status':'thirteen_exact_NPORT_index_pages_archived_parse_pending' if error is None and len(receipts)==13 else 'partial_index_transport_needs_repair','error':error,'GET_attempts':state['GET_attempts'],'HTTP200_pages':sum(x.get('http_status')==200 and x.get('curl_exit_code')==0 for x in receipts),'nas_archive_sha256':digest,'nas_archive_bytes':evidence['size_bytes'],'nas_remote_and_scoped_restore_verified':True,'primary_document_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'transport_result.json').write_text(json.dumps(result,indent=2)+'\n')
print({k:result[k] for k in ('status','GET_attempts','HTTP200_pages','nas_archive_sha256')})


{'status': 'thirteen_exact_NPORT_index_pages_archived_parse_pending', 'GET_attempts': 13, 'HTTP200_pages': 13, 'nas_archive_sha256': 'dfdf9c63a15972786078b8c2058e1c677c9c173f4e66dff47cd2d0109a21abda'}


## 中文结论

13 条 SOXX series-specific 2019–2022 NPORT-P filing index URL 一次取得，全部 HTTP 200；原始页面与响应头 NAS 归档、远端哈希和定点恢复通过，归档 SHA-256 `dfdf9c63a15972786078b8c2058e1c677c9c173f4e66dff47cd2d0109a21abda`。无 XML、收益、因子、拟合或账户。
